## Getting Started with LangChain on Databricks

### Installing Utilities and Libraries

In [0]:
%pip install databricks-langchain==0.12.1 langchain-community==0.4.1 langchain-experimental==0.4.1

### Restarting the Python Kernel

In [0]:
dbutils.library.restartPython()

### Invoking a Chat Model with Databricks

In [0]:
import json
from databricks_langchain import ChatDatabricks

chat_model = ChatDatabricks(
    endpoint="databricks-claude-sonnet-4-5",
    temperature=0.1,
    max_tokens=250,
)
response = chat_model.invoke("How to use Databricks?")
json_response = response.to_json()
display(json_response['kwargs']['content'])

### Using Mosaic AI Vector Search Index with LangChain

In [0]:
from databricks_langchain import DatabricksVectorSearch

vector_store = DatabricksVectorSearch(index_name="YOUR_UNITY_CATALOG_NAME.SCHEMA.VECTOR_SEARCH_INDEX_NAME")
retriever = vector_store.as_retriever(search_kwargs={"k": 5})
retriever.invoke("what is the carbonops ESG intelligence Model (CEIM)")

### Creating a Spark Dataframe Agent with LangChain

#### Saving the CSV File in the Unity Catalog Volume

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS genai_lab

In [0]:
import requests

# Define the current catalog
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Define the base path using the current catalog
volume_base = f"/Volumes/{catalog_name}/default/genai_lab"

# List of files to download
files = ["restaurant_reviews.csv"]

# Download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DatabricksGenAIEngineer/refs/heads/main/GenAI_Fundamentals/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

#### Creating the Spark Dataframe Agent

In [0]:
from langchain_experimental.agents.agent_toolkits import create_spark_dataframe_agent
from databricks_langchain import ChatDatabricks

df = spark.read.csv(f"/Volumes/{catalog_name}/default/genai_lab/restaurant_reviews.csv", header=True, inferSchema=True)
display(df)

chat_model = ChatDatabricks(
    endpoint="databricks-claude-sonnet-4-5",
    temperature=0.1,
    max_tokens=250,
)

agent = create_spark_dataframe_agent(llm=chat_model, df=df, verbose=True, allow_dangerous_code=True)


In [0]:
agent.run("total number of reviews?")

In [0]:
agent.run("what is the review for cheesecake?")

In [0]:
agent.run("What is the review given on tacos?")